# Week 11 — Build one transparent research-agent tool call

**Research task:** Let a model request one search of a supplied course corpus, then preserve the tool request, source return and drafted claim.

**Python introduced:** functions as tools, arguments, returned dictionaries, conditional dispatch and provenance records.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session11/session11_agentic_provenance.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.


In [ ]:
SESSION = "session11"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib.util as setup_importlib
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath("/content/GenAI_Soc2026")
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The course repository root could not be found. Start Jupyter from the "
            "GenAI_Soc2026 folder with: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Choose a route and store a small local source corpus

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.


In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
course_corpus = [
    {"id":"s1","title":"Replication for language models","excerpt":"A rerun must preserve prompts, model access, settings and outputs."},
    {"id":"s2","title":"Research agents and long tasks","excerpt":"Errors can enter during search, extraction, coding and synthesis."},
    {"id":"s3","title":"Synthetic survey respondents","excerpt":"Aggregate similarity does not establish individual correspondence."},
]


## Define the one tool available to the agent

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
def search_course_corpus(query):
    query_words = query.lower().split()
    matches = []
    for source in course_corpus:
        searchable = (source["title"] + " " + source["excerpt"]).lower()
        if any(word in searchable for word in query_words):
            matches.append(source)
    return {"query":query,"matches":matches}

## Ask the model which search to make

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
request_schema = {
    "type":"object","properties":{
        "tool_name":{"type":"string","enum":["search_course_corpus"]},
        "query":{"type":"string"}},
    "required":["tool_name","query"],"additionalProperties":False,
}
research_question = "Where can errors enter during a long research-agent task?"
messages = [{"role":"user","content":(
    "Choose the one available tool and a short query for this research question: " + research_question
)}]
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(model=HOSTED_MODEL,messages=messages,temperature=0,response_format={"type":"json_schema","json_schema":{"name":"tool_request","strict":True,"schema":request_schema}})
    request_raw = response.choices[0].message.content
else:
    response = ollama.chat(model=LOCAL_MODEL,messages=messages,format=request_schema,options={"temperature":0})
    request_raw = response.message.content
tool_request = json.loads(request_raw)
print("Raw tool request:", request_raw)

## Dispatch the named tool and inspect its returned record

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
if tool_request["tool_name"] == "search_course_corpus":
    tool_result = search_course_corpus(tool_request["query"])
print("Tool result:", tool_result)

## Return the evidence to the model and request one bounded claim

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
claim_schema = {"type":"object","properties":{"claim":{"type":"string"},"source_id":{"type":"string"}},"required":["claim","source_id"],"additionalProperties":False}
claim_messages = [{"role":"user","content":(
    "Answer with one bounded claim and one source_id using only this tool result: " + json.dumps(tool_result)
)}]
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(model=HOSTED_MODEL,messages=claim_messages,temperature=0,response_format={"type":"json_schema","json_schema":{"name":"source_claim","strict":True,"schema":claim_schema}})
    claim_raw = response.choices[0].message.content
else:
    response = ollama.chat(model=LOCAL_MODEL,messages=claim_messages,format=claim_schema,options={"temperature":0})
    claim_raw = response.message.content
claim = json.loads(claim_raw)
trajectory = {"question":research_question,"tool_request":tool_request,"tool_result":tool_result,"claim":claim}
print("Trajectory:", trajectory)

# ONE CHANGE: ask "What must be saved to reproduce an LLM result?"

## Methodological check

A source ID is valid only if it resolves to a returned source and the excerpt supports the claim. Repetition by several agents would not create independent corroboration.
## Recording

Change the question and trace question → raw tool request → function argument → returned source record → raw claim → trajectory. Reject an unresolved or unsupported claim.